# Notebook 04: Multi-Model Comparison

**Purpose:** Train RF, DT, XGBoost, and LR on both datasets. Run 5-fold stratified CV. Compute all metrics on the test set. Save trained models and a master results table.

**Outputs:** `results/models/`, `results/master_results_all.csv`

In [1]:
import os
# Use the repository root as the working directory, whether this notebook is
# launched from the repo root or from the notebooks/ folder.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import pandas as pd
import numpy as np
import joblib
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    confusion_matrix, accuracy_score, matthews_corrcoef,
    balanced_accuracy_score, average_precision_score)
from xgboost import XGBClassifier

os.makedirs('results/models', exist_ok=True)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Libraries loaded.')

Libraries loaded.


In [2]:
ugr_train = pd.read_csv('data/processed/ugr_train.csv')
ugr_test  = pd.read_csv('data/processed/ugr_test.csv')
cic_train = pd.read_csv('data/processed/cic_train.csv')
cic_test  = pd.read_csv('data/processed/cic_test.csv')

FEAT_UGR = [c for c in ugr_train.columns if c != 'Prediction']
FEAT_CIC = [c for c in cic_train.columns if c not in ['label', 'label_binary']]

X_tr_ugr = ugr_train[FEAT_UGR].values
y_tr_ugr = ugr_train['Prediction'].values
X_te_ugr = ugr_test[FEAT_UGR].values
y_te_ugr = ugr_test['Prediction'].values

X_tr_cic = cic_train[FEAT_CIC].values
y_tr_cic = cic_train['label_binary'].values
X_te_cic = cic_test[FEAT_CIC].values
y_te_cic = cic_test['label_binary'].values

print(f'UGR: train={X_tr_ugr.shape}, test={X_te_ugr.shape}')
print(f'CIC: train={X_tr_cic.shape}, test={X_te_cic.shape}')

UGR: train=(71887, 49), test=(17972, 49)
CIC: train=(159874, 46), test=(39995, 46)


In [3]:
ugr_pos_weight = (y_tr_ugr == 0).sum() / (y_tr_ugr == 1).sum()
cic_pos_weight = (y_tr_cic == 0).sum() / (y_tr_cic == 1).sum()

MODELS = {
    'RandomForest': {
        'ugr': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1),
        'cic': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1)
    },
    'DecisionTree': {
        'ugr': DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
        'cic': DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced')
    },
    'XGBoost': {
        'ugr': XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=ugr_pos_weight,
                            use_label_encoder=False, eval_metric='logloss', verbosity=0),
        'cic': XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=cic_pos_weight,
                            use_label_encoder=False, eval_metric='logloss', verbosity=0)
    },
    'LogisticRegression': {
        'ugr': LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced',
                                  max_iter=1000, solver='liblinear'),
        'cic': LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced',
                                  max_iter=1000, solver='liblinear')
    }
}
print('Models defined.')
print(f'UGR scale_pos_weight: {ugr_pos_weight:.2f}')
print(f'CIC scale_pos_weight: {cic_pos_weight:.2f}')

Models defined.
UGR scale_pos_weight: 3.39
CIC scale_pos_weight: 0.02


In [4]:
def compute_metrics(y_true, y_pred, y_prob, model_name, dataset):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    npv  = tn / (tn + fn) if (tn + fn) > 0 else 0
    fpr  = fp / (fp + tn) if (fp + tn) > 0 else 0
    return {
        'dataset': dataset, 'model': model_name,
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_attack': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'f1_benign': f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'accuracy': accuracy_score(y_true, y_pred),
        'auc_roc': roc_auc_score(y_true, y_prob),
        'pr_auc': average_precision_score(y_true, y_prob),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'balanced_acc': balanced_accuracy_score(y_true, y_pred),
        'specificity': spec, 'npv': npv, 'fpr': fpr,
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn)
    }

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print('Metric function and CV defined.')

Metric function and CV defined.


In [5]:
# Train and evaluate all models
all_results = []

datasets = [
    ('UGRansome2024', 'ugr', X_tr_ugr, y_tr_ugr, X_te_ugr, y_te_ugr),
    ('CICIoT2023',    'cic', X_tr_cic, y_tr_cic, X_te_cic, y_te_cic)
]

for ds_name, ds_key, X_tr, y_tr, X_te, y_te in datasets:
    print(f'\n=== {ds_name} ===')
    for model_name, model_dict in MODELS.items():
        model = model_dict[ds_key]
        t0 = time.perf_counter()
        model.fit(X_tr, y_tr)
        train_time = time.perf_counter() - t0

        # CV F1
        cv_scores = cross_val_score(model, X_tr, y_tr, cv=cv,
                                    scoring='f1_macro', n_jobs=-1)

        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1]

        metrics = compute_metrics(y_te, y_pred, y_prob, model_name, ds_name)
        metrics['training_time_sec'] = round(train_time, 3)
        metrics['cv_f1_mean'] = round(cv_scores.mean(), 4)
        metrics['cv_f1_std']  = round(cv_scores.std(),  4)
        all_results.append(metrics)

        # Save model
        model_path = f'results/models/{ds_key}_{model_name}.joblib'
        joblib.dump(model, model_path, compress=3)

        # Save predictions
        pred_df = pd.DataFrame({'y_true': y_te, 'y_pred': y_pred, 'y_prob': y_prob})
        pred_df.to_csv(f'results/baselines/{ds_key}_{model_name}_predictions.csv', index=False)

        print(f'  {model_name}: F1={metrics["f1_macro"]:.4f}, '
              f'CV={metrics["cv_f1_mean"]:.4f}+/-{metrics["cv_f1_std"]:.4f}, '
              f'train={train_time:.1f}s')


=== UGRansome2024 ===


  RandomForest: F1=0.9911, CV=0.9921+/-0.0006, train=2.4s


  DecisionTree: F1=0.9927, CV=0.9931+/-0.0007, train=0.3s


  XGBoost: F1=0.9944, CV=0.9948+/-0.0008, train=0.7s


  LogisticRegression: F1=0.8951, CV=0.8880+/-0.0034, train=0.5s

=== CICIoT2023 ===


  RandomForest: F1=0.9622, CV=0.9617+/-0.0038, train=5.1s


  DecisionTree: F1=0.9493, CV=0.9525+/-0.0028, train=0.8s


  XGBoost: F1=0.9464, CV=0.9414+/-0.0012, train=1.3s


  LogisticRegression: F1=0.8283, CV=0.8379+/-0.0048, train=6.9s


In [6]:
master = pd.DataFrame(all_results)
master.to_csv('results/master_results_all.csv', index=False)
print('Saved master_results_all.csv')

print('\n=== Summary table ===')
print(master[['dataset','model','f1_macro','auc_roc','mcc','training_time_sec']].to_string())

Saved master_results_all.csv

=== Summary table ===
         dataset               model  f1_macro   auc_roc       mcc  training_time_sec
0  UGRansome2024        RandomForest  0.991080  0.999857  0.982163              2.358
1  UGRansome2024        DecisionTree  0.992731  0.993218  0.985462              0.304
2  UGRansome2024             XGBoost  0.994416  0.999939  0.988871              0.659
3  UGRansome2024  LogisticRegression  0.895109  0.948168  0.792835              0.478
4     CICIoT2023        RandomForest  0.962162  0.999530  0.924340              5.052
5     CICIoT2023        DecisionTree  0.949307  0.941429  0.898772              0.833
6     CICIoT2023             XGBoost  0.946426  0.999349  0.897493              1.263
7     CICIoT2023  LogisticRegression  0.828337  0.985447  0.683213              6.871


In [7]:
print('\n=== Best per dataset ===')
for ds in master['dataset'].unique():
    sub = master[master['dataset'] == ds]
    best = sub.loc[sub['f1_macro'].idxmax()]
    print(f'{ds}: {best["model"]} F1={best["f1_macro"]:.4f} AUC={best["auc_roc"]:.4f}')


=== Best per dataset ===
UGRansome2024: XGBoost F1=0.9944 AUC=0.9999
CICIoT2023: RandomForest F1=0.9622 AUC=0.9995


## Summary

**Purpose:** Train and evaluate four classifiers on both datasets.

**Method:** Random Forest (100 trees), Decision Tree, XGBoost, and Logistic Regression trained with class_weight='balanced' (or scale_pos_weight for XGBoost). 5-fold stratified cross-validation on training data. Final evaluation on held-out test set. All hyperparameters fixed per EXPERIMENT_CONFIG.md.

**Key findings:** See master_results_all.csv and printed summary above.